In [1]:
import math
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [2]:
#calculate passage number when I ran out of passage 5 vials ;;
def passage_number(time):
    base_vial = 6
    if time > 2:
        base_vial = base_vial-1 
    pnum = base_vial + (3 * time)
    return pnum



def well_namer(row, col):
    well_name = str(chr(ord('@')+ row)) + str(col).rjust(2, '0')  #make the number have a left align, adding a zero
    return well_name



def extract_number(string, ndigits=None):
    if ndigits is not None:
        match = re.search(fr"\d{{{ndigits}}}", string)
    else:
        match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return None
    
def extract_repnumber(string, ndigits=None):
    if ndigits is not None:
        match = re.search(rf"rep\d{{{ndigits}}}", string)
    else:
        match = re.search(r"rep\d+", string)
    if match:
        return int(match.group().replace("rep", ""))
    else:
        return None

print(extract_number("20240313_rep01_map", 2))
print(extract_repnumber("20240313_rep03_map", 2))


20
3


In [3]:
# Get 96-well plate CSV file from benchiling/Notion NOTE: don't initialze an empty df, use a list and convert after.

plate_path = (
    "plate_metadata/"  #'/home/mattiazzilab/Documents/Allie_Scripts/May 8 seeding.csv'
)

export_path = "plate_metadata/"  #'/mnt/bigdisk1/Allie_S/Replicative_Age_Project/Data Mining/metadata/'


def load_plate_df(path):
    raw_plate_df = pd.read_csv(path, header=0, usecols=range(1, 13)).dropna(
        axis=1, how="all"
    )
    plate_df = raw_plate_df.dropna(axis=0, thresh=2)
    print(plate_df.shape)  # Checks if correct number of rows and columns

    return plate_df


def make_map_df(condition_cols, aux_cols):
    """
    Return a df with the required columns
    """
    main_columns = [
        "Metadata_Plate",
        "Folder_Path",
        "Metadata_Well",
        "Metadata_WellRow",
        "Metadata_WellColumn",
        "Metadata_Field",
        "Metadata_RowColFieldCode",
        "Staining",
    ]
    df_cols = main_columns + condition_cols + aux_cols
    plate_map_df = pd.DataFrame(columns=df_cols)
    print(plate_map_df.columns)
    return plate_map_df


# plate_df.head(13)
# print(columns_row)
condition_cols = [
    "PassageNumber",
    "Lineage",
    "AgeGroup",
]
aux_cols = [
    "Drug",
    "FlaggedBatch",
    "LineageNumber",
    "TimepointName",
]
make_map_df(condition_cols, aux_cols)

Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')


,Metadata_Plate,Folder_Path,Metadata_Well,Metadata_WellRow,Metadata_WellColumn,Metadata_Field,Metadata_RowColFieldCode,Staining,PassageNumber,Lineage,AgeGroup,Drug,FlaggedBatch,LineageNumber,TimepointName


In [4]:
def conditional_ops(condition_cols, well_metadata, aux_cols=[]):
    updates_dict = {}
    text = well_metadata.split(" ")
    # use regex to extract the numerical bits e.g.  grab the number after the B in the serial passage batch
    all_cols = condition_cols+aux_cols
    try:
        for condition in all_cols:
            if condition == "PassageNumber":
                # passage_number(Int(time)) - use the function if you don;t have passage number in the table
                updates_dict[condition] = extract_number(text[0]) 
            elif condition == "TreatmentGroup":
                updates_dict[condition] = text[0]
            elif condition == "LineageNumber":
                updates_dict[condition] = extract_number(text[1].split("-")[0])  
            elif condition == "Lineage":
                updates_dict[condition] = text[1]
            elif condition == "AgeGroup":
                # time used to mean age group - deprecated term but still used in code
                updates_dict[condition] = extract_number(text[2]) 
            elif condition == "Drug":
                # grab drug, name an
                if ("_" in text[0]):  
                    # if there is an underscore than the well is drug-treated for this group
                    drug = text[0].split("_")[0]
                    updates_dict[condition] = drug
                    if "Lineage" in updates_dict: 
                        updates_dict["Lineage"] = f"{drug}_" + updates_dict["Lineage"]
                    # grab the drug name before the underscore
                else:
                    updates_dict[condition] = None
            elif condition == "FlaggedBatch":
                # flag passage if we have "Flagged" in the serial passage batch
                updates_dict[condition] = "Flagged" in text[0]
            elif condition == "TimepointName":
                updates_dict[condition] = text[0] + " " + text[1] + " " + text[2]
            elif condition == "ShortStaining":
                new_text = []
                for term in text:
                    term = term.removesuffix("INK4A")
                    term = term.removesuffix("CIP1WAF1")
                    if term != "+":
                        new_text.append(term)
                stains = "_".join(new_text[len(condition_cols) : len(new_text)])
                updates_dict[condition] = stains
        return updates_dict
    except IndexError as e:
        print(e, "Careful, this went out of bounds. Did you keep a condition in all_cols that is present in the text on one plate but not the others? e.g. age group, lineage, etc")

In [5]:
def export_platemap_csv(plate_df, plate_name, export_path, condition_cols, aux_cols=[], file_path="", folder_path = "", plate_number = None):
    plate_map_df = make_map_df(condition_cols,aux_cols)
    columns_row = plate_df.columns  # get_columns_row(plate_df)
    # little thing to get the parent folder with the full plate name and number
    if file_path:
        file_path = Path(file_path)
        if folder_path == "":
            folder_path = file_path.parent
            if re.search(r'\d{1,2}', file_path.name):
                plate_number = extract_repnumber(file_path.name, ndigits=2)
                
            
    for index, data in plate_df.iterrows():
        row = data.to_list()
        for count, value in enumerate(row):
            curr_well = value
            if pd.isna(curr_well):
                continue
            row_list = []
            # get string data and label of row in df (e.g. col1, text= R1T0 EAA1-488 Tfn-647)
            # regex to separate into different variables
            # then add them to dict with their respective col index (label) and index of the row in the column (column.index)
            for i in range(40):
                row_entry = {}

                row_index = index + 1  # Make it 1-indexed

                column_index = columns_row[count]  # Use header row for column index
                # print(column_index)

                if ~np.isnan(row_index):
                    well_name = well_namer(row_index, column_index)
                else:
                    well_name = "Empty"
                    continue
                
                # information for the field is just 1-40, nothing else changes
                field = i + 1
                rowcolfield = f"r{str(row_index).rjust(2, '0')}c{str(column_index).rjust(2, '0')}f{str(field).rjust(2, '0')}"
                
                # seperate well metadata by space
                text = curr_well.split(" ")
                
                #stains come after all the conditions
                n_conditions = len(condition_cols)
                stains = " ".join(
                    text[n_conditions : len(text)]
                ) 
                stains=stains.replace("+=","")
                
                row_entry.update(
                    {
                        "Metadata_Plate" : plate_name,
                        "Folder_Path" : folder_path,
                        "Plate_Number" : plate_number,
                        "Metadata_Well": well_name,
                        "Metadata_WellRow": row_index,
                        "Metadata_WellColumn": column_index,
                        "Metadata_Field": field,
                        "Staining": stains.strip(),
                        "Metadata_RowColFieldCode":rowcolfield,
                    }
                )
                conditional_entries = conditional_ops(condition_cols, curr_well, aux_cols)
                row_entry.update(conditional_entries)

                row_list.append(row_entry)

            rows = pd.DataFrame(row_list)
            plate_map_df = pd.concat([plate_map_df, rows], ignore_index=True)
            # column_index = column_index+1
    plate_map_df.to_csv(os.path.join(export_path, f"{plate_name}_map.csv"), index=False)

In [6]:

for root, dirs, files in os.walk(plate_path):
    for filename in files:
        if filename.endswith(".csv") and "map" not in filename and "Pilot" and "replacements" not in filename:
            file_path = os.path.join(root,filename)
            plate_df = load_plate_df(os.path.abspath(file_path))
            display(plate_df)
            export_path = os.path.abspath(root)
            print(f"Exporting {filename} to {export_path}")
            plate_name = filename.split(".")[0]
            export_platemap_csv(plate_df, plate_name, export_path, condition_cols, aux_cols=aux_cols, file_path=file_path)
            

(6, 10)


,2,3,4,5,6,7,8,9,10,11
1,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647
2,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647
3,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647
4,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647
5,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647
6,P20 LIN1-0B-b1 AG4 LAMP1-488 + MitoRed + WGA-C...,P17 LIN2-0A-b1 AG3 LAMP1-488 + MitoRed + WGA-C...,P14 LIN3-0A-b2 AG2 LAMP1-488 + MitoRed + WGA-C...,P12 LIN4-0B-b2 AG1 LAMP1-488 + MitoRed + WGA-C...,P10 LIN5-0B-b3 AG0 LAMP1-488 + MitoRed + WGA-C...,P20 LIN1-0B-b1 AG4 EEA1-488 + ConA-594 + Tfn-647,P17 LIN2-0A-b1 AG3 EEA1-488 + ConA-594 + Tfn-647,P14 LIN3-0A-b2 AG2 SPB3 AG2 P11 EEA1-488 + Con...,P12 LIN4-0B-b2 AG1 EEA1-488 + ConA-594 + Tfn-647,P10 LIN5-0B-b3 AG0 EEA1-488 + ConA-594 + Tfn-647


Exporting 20240313_rep01.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20240313_rep01_metadata
Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')
(8, 6)


,2,3,4,5,6,7
0,P25 LIN2-0A-b1-s1 AG5 Phalloidin-647 + ConA-594,P22 LIN3-0A-b2-s1 AG4 Phalloidin-647 + ConA-594,P20 LIN4-0B-b2A-s1 AG3 Phalloidin-647 + ConA-5...,P18 LIN5-0B-b3-s1 AG2 Phalloidin-647 + ConA-594,P16 LIN6-0C-b1-s1 AG1 Phalloidin-647 + ConA-594,P13 LIN7-0C-b3 AG0 Phalloidin-647 + ConA-594
1,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
2,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
3,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
4,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
5,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
6,P25 LIN2-0A-b1-s1 AG5 LAMP1-488 + MitoRed,P22 LIN3-0A-b2-s1 AG4 LAMP1-488 + MitoRed,P20 LIN4-0B-b2A-s1 AG3 LAMP1-488 + MitoRed,P18 LIN5-0B-b3-s1 AG2 LAMP1-488 + MitoRed,P16 LIN6-0C-b1-s1 AG1 LAMP1-488 + MitoRed,P13 LIN7-0C-b3 AG0 LAMP1-488 + MitoRed
7,P25 LIN2-0A-b1-s1 AG5 Phalloidin-647 + WGA-647...,P22 LIN3-0A-b2-s1 AG4 Phalloidin-647 + WGA-647...,P20 LIN4-0B-b2A-s1 AG3 Phalloidin-647 + WGA-64...,P18 LIN5-0B-b3-s1 AG2 Phalloidin-647 + WGA-647...,P16 LIN6-0C-b1-s1 AG1 Phalloidin-647 + WGA-647...,P13 LIN7-0C-b3 AG0 Phalloidin-647 + WGA-647 + ...


Exporting 20241018_rep03.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20241018_rep03_metadata
Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')
(6, 10)


...

Exporting 20250410_rep06.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata
Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')
(6, 12)


,1,2,3,4,5,6,7,8,9,10,11,12
1,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...
2,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...
3,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...
4,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...
5,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...
6,Doxo_P19 LIN12-0C-b2-s2 AG10 LAMP1-488 + MitoR...,P34 LIN3-0A-b2-s1-ss2 AG9 LAMP1-488 + MitoRed ...,P33 LIN3-0A-b2-s2-ss1-sss1 AG8 LAMP1-488 + Mit...,P29 LIN2-0A-b1-s1-ss1 AG7 LAMP1-488 + MitoRed ...,P25 LIN8-0B-b2B-s1-ss1 AG6 LAMP1-488 + MitoRed...,P25 LIN8-0B-b2B-s1-ss2 AG5 LAMP1-488 + MitoRed...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P24 LIN7-0C-b3-s2 AG3 LAMP1-488 + MitoRed + Ph...,P21 LIN11-0C-b5 AG2 LAMP1-488 + MitoRed + Phal...,P20 LIN12-0C-b2-s2 AG1 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...,P13 LIN11-0C-b5-s1 AG0 LAMP1-488 + MitoRed + P...


Exporting 20250501_rep07.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata
Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')
(6, 12)


,1,2,3,4,5,6,7,8,9,10,11,12
1,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p16INK4A-594 1:800 + Phall...,Doxo_P18 LIN13-0C-b2-s3 AG9 p16INK4A-594 1:800...
2,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p16INK4A-594 1:1600 + Phal...,Doxo_P18 LIN13-0C-b2-s3 AG9 p16INK4A-594 1:160...
3,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p16INK4A-594 1:2600 + Phal...,Doxo_P18 LIN13-0C-b2-s3 AG9 p16INK4A-594 1:260...
4,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p16INK4A-594 1:3600 + Phal...,Doxo_P18 LIN13-0C-b2-s3 AG9 p16INK4A-594 1:360...
5,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p16INK4A-594 1:3600 LMNB1 ...,Doxo_P18 LIN13-0C-b2-s3 AG9 p16INK4A-594 1:360...
6,Doxo_P18 LIN13-0C-b2-s3 AG9 LAMP1-488 + MitoRe...,P33 LIN3-0A-b2-s1-ss1 AG8 LAMP1-488 + MitoRed ...,P32 LIN3-0A-b2-s1-ss1-sss1 AG7 LAMP1-488 + Mit...,P29 LIN3-0A-b2-s1-ss2 AG6 LAMP1-488 + MitoRed ...,P27 LIN3-0A-b2-s2-ss1-sss1 AG5 LAMP1-488 + Mit...,P24 LIN6-0C-b1-s2 AG4 LAMP1-488 + MitoRed + Ph...,P19 LIN8-0B-b2B-s1-ss1 AG3 LAMP1-488 + MitoRed...,P19 LIN8-0B-b2B-s1-ss2 AG2 LAMP1-488 + MitoRed...,P15 LIN6-0C-b1-s2 AG1 LAMP1-488 + MitoRed + Ph...,P11 LIN11-0C-b5 AG0 LAMP1-488 + MitoRed + Phal...,P11 LIN11-0C-b5 AG0 p21-594 1:6000 + Phalloidi...,Doxo_P18 LIN13-0C-b2-s3 AG9 p21-594 1:600 + Ph...


Exporting 20250328_rep05.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata
Index(['Metadata_Plate', 'Folder_Path', 'Metadata_Well', 'Metadata_WellRow',
       'Metadata_WellColumn', 'Metadata_Field', 'Metadata_RowColFieldCode',
       'Staining', 'PassageNumber', 'Lineage', 'AgeGroup', 'Drug',
       'FlaggedBatch', 'LineageNumber', 'TimepointName'],
      dtype='object')


In [7]:
#for the pilot one
for root, dirs, files in os.walk(plate_path):
    for filename in files:
        if (
            filename.endswith(".csv")
            and "map" not in filename
            and "Pilot" in filename
        ):
            file_path = os.path.join(root, filename)
            plate_df = load_plate_df(os.path.abspath(file_path))
            display(plate_df)
            export_path = os.path.abspath(root)
            print(f"Exporting {filename} to {export_path}")
            plate_name = filename.split(".")[0]
            
            pilot_conditions = ["TreatmentGroup"]
            pilot_aux = ["ShortStaining"]
            
            export_platemap_csv(
                plate_df, plate_name, export_path, condition_cols=pilot_conditions, aux_cols=pilot_aux
            )
